In [ ]:
!pip install -r .\requirements.txt

# Modele

### Chat Model
Jeżeli nie masz pobranego modelu, wykonaj kod !ollama pul... aby pobrać model z .env

#### Non-Streaming

In [ ]:
from app.core import LLM_MODEL
!ollama pull {LLM_MODEL}

In [ ]:
from app.core import ChatModel, LLM_MODEL

chat = ChatModel(model=LLM_MODEL, system="You are a helpful assistant", memory=False)
r = chat.ask("Napisz małą rozprawkę o Szekspirze", think=False)

print(r)

#### Streaming
W .ipnyb streaming nie zadziała

In [ ]:
from app.core import ChatModel, LLM_MODEL

chat = ChatModel(model=LLM_MODEL, system="You are a helpful assistant", memory=False)
for piece in chat.ask_stream("Napisz małą rozprawkę o Szekspirze", think=False):
    print(piece, end="", flush=True)

### Embed Model
Jeżeli nie masz pobranego modelu, wykonaj kod !ollama pul... aby pobrać model z .env

In [ ]:
from app.core import EMBED_MODEL
!ollama pull {EMBED_MODEL}

In [ ]:
from app.core import EmbedModel, EMBED_MODEL, EMBED_MODEL_DIM

embed = EmbedModel(model=EMBED_MODEL, dim=EMBED_MODEL_DIM)
e = embed.encode("Lorem ipsum dolor sit amed")

print(e)

# RAG
Stwórz bazy danych tą komendą jeżeli jeszcze tego nie zrobiłeś

In [1]:
!docker compose -f database/docker-compose.yml up -d

time="2026-08-06T15:16:08+02:00" level=warning msg="The \"GRAPH_DB_PASSWORD\" variable is not set. Defaulting to a blank string."
 Image neo4j:5 Pulling 
 4f4fb700ef54 Pulling fs layer 0B
 26c307b5e35a Pulling fs layer 0B
 79b455972dd2 Pulling fs layer 0B
 e35aaf9bcce1 Pulling fs layer 0B
 d603e74598f9 Pulling fs layer 0B
 b47eda4dae65 Download complete 0B
 b3a634858b6b Downloading 1.049MB
 4f4fb700ef54 Downloading 32B
 b3a634858b6b Downloading 1.049MB
 4f4fb700ef54 Download complete 0B
 26c307b5e35a Downloading 1.049MB
 e35aaf9bcce1 Download complete 0B
 b3a634858b6b Downloading 2.097MB
 26c307b5e35a Downloading 1.049MB
 26c307b5e35a Downloading 1.049MB
 b3a634858b6b Downloading 2.097MB
 26c307b5e35a Downloading 2.097MB
 b3a634858b6b Downloading 3.146MB
 b3a634858b6b Downloading 3.146MB
 26c307b5e35a Downloading 2.097MB
 d603e74598f9 Downloading 1.049MB
 b3a634858b6b Downloading 3.146MB
 26c307b5e35a Downloading 2.097MB
 79b455972dd2 Downloading 1.049MB
 d603e74598f9 Downloading 1.049

### Graph RAG

#### Procedural Graph Database Set-Up

#### LLM Graph Database Set-Up

In [ ]:
from app.core import GRAPH_MODEL
!ollama pull {GRAPH_MODEL}

In [ ]:
from app.graph import initialize_knowledge_graph, initialize_graph_driver

initialize_graph_driver()
initialize_knowledge_graph()

In [ ]:
from app.ingest import load_knowledge, check_for_duplicates
from app.schema import prepare_for_vector_embedding, prepare_for_lexical_search, prepare_for_prompt

documents = load_knowledge("./knowledge")
check_for_duplicates(documents)

document_string: str = '\n\n'.join(
    [prepare_for_vector_embedding(document) for document in documents]
)

In [ ]:
from app.graph import build_graph_with_ollama, print_graph, knowledge_graph, graph_driver
from app.core import GRAPH_MODEL

build_graph_with_ollama(model=GRAPH_MODEL, documents=document_string)

print_graph()

input("[ENTER], aby zsynchronizować do bazy danych...")

knowledge_graph.sync(driver=graph_driver)

print("\nGotowe!")

In [ ]:
!start http://localhost:7474